# Scalp2 Backtest Optimization

This notebook performs a deep analysis of the walk-forward validation results. It simulates the exact `TradeManager` logic over historical predictions to find the optimal **ATR Filter** and **Confidence Thresholds**, and visualizes performance by **Time of Day** and **Market Regime**.

## 1. Setup & Load Data

In [ ]:
# Install dependencies\n!pip install -q xgboost ccxt PyWavelets hmmlearn numba scikit-learn pyyaml \\n    tqdm pyarrow matplotlib seaborn

In [ ]:
from google.colab import drive\ndrive.mount('/content/drive')\n\nimport os, sys, json\nREPO_DIR = '/content/scalp2_repo'\nif os.path.exists(os.path.join(REPO_DIR, '.git')):\n    !git -C {REPO_DIR} pull --ff-only\nelse:\n    !git clone https://github.com/sergul74/Scalp2.git {REPO_DIR}\n\nsys.path.insert(0, REPO_DIR)\n\nimport logging\nlogging.basicConfig(level=logging.INFO)\n

In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm.auto import tqdm

from scalp2.config import load_config
from scalp2.execution.trade_manager import TradeManager, TradeState, TradeStatus

plt.style.use('dark_background')
sns.set_palette('deep')

In [ ]:
# Set DATA_DIR to where your Colab processed data is (e.g. Google Drive)
DATA_DIR = '/content/drive/MyDrive/scalp2/data/processed'
config = load_config(f'{REPO_DIR}/config.yaml')

# 1. Load the historical features dataset (that contains atr_14, regime_probs, etc)
print("Loading dataset...")
try:
    df = pd.read_parquet(f'{DATA_DIR}/BTC_USDT_labeled.parquet')
except:
    print("Parquet not found. Ensure the dataset is built.")
    
# 2. Load the walk-forward predictions (which has probabilities for each timestamp)
print("Loading predictions...")
try:
    with open(f'{DATA_DIR}/wf_predictions.pkl', 'rb') as f:
        wf_predictions = pickle.load(f)
except:
    print("Predictions pickle not found! Please run Notebook 06 to generate wf_predictions.pkl.")

## 2. Core Simulation Engine

In [ ]:
def simulate_sweep(df, wf_predictions, config, min_atr_percentile, confidence_threshold):
    """Replays TradeManager over Walk-Forward Predictions."""
    trade_mgmt_cfg = config.execution.trade_management
    label_cfg = config.labeling
    trade_mgr = TradeManager(trade_mgmt_cfg, label_cfg.max_holding_bars)
    
    # 1. Flatten all Walk-Forward predictions into aligned arrays
    p_short_list = []
    p_long_list = []
    regime_probs_list = []
    bar_indices = []
    
    seq_len = config.model.seq_len
    for fold_data in wf_predictions:
        preds = fold_data['test_probabilities']
        regime = fold_data.get('regime_probs', np.full((len(preds), 3), 1/3, dtype=np.float32))
        offset = fold_data['test_start'] + seq_len
        
        valid_len = min(len(preds), len(df) - offset)
        if valid_len <= 0: continue
        
        p_short_list.append(preds[:valid_len, 0])
        p_long_list.append(preds[:valid_len, 2])
        regime_probs_list.append(regime[:valid_len])
        bar_indices.append(np.arange(offset, offset + valid_len))
        
    if not bar_indices: return pd.DataFrame()
    
    p_short = np.concatenate(p_short_list)
    p_long = np.concatenate(p_long_list)
    regime_probs = np.concatenate(regime_probs_list)
    bars = np.concatenate(bar_indices)
    
    # Precompute global indicators to avoid gaps
    atr_pcts_full = df['atr_14'].rolling(96).rank(pct=True).fillna(1.0).values if 'atr_14' in df else np.full(len(df), 1.0)
    adxs_full = df['adx'].values if 'adx' in df else np.full(len(df), 999.0)
    
    df_sim = df.iloc[bars]
    
    active_trade = None
    pending_signal = None
    daily_trades = 0
    prev_date = None
    trades_log = []
    
    # Convert to fast numpy arrays for iteration
    times = df_sim.index
    highs = df_sim['high'].values
    lows = df_sim['low'].values
    closes = df_sim['close'].values
    opens = df_sim['open'].values
    atrs = df_sim['atr_14'].values if 'atr_14' in df_sim else np.zeros(len(df_sim))
    
    atr_pcts = atr_pcts_full[bars]
    adxs = adxs_full[bars]
    
    for i in range(len(df_sim) - 1):
        cur_time = times[i]
        cur_date = cur_time.date()
        
        if cur_date != prev_date:
            daily_trades = 0
            prev_date = cur_date
            
        # 1. Update Active Trade
        if active_trade is not None:
            is_choppy = regime_probs[i, 2] > config.regime.choppy_threshold
            active_trade = trade_mgr.update(active_trade, highs[i], lows[i], closes[i], is_choppy)
            
            if active_trade.status not in (TradeStatus.OPEN, TradeStatus.PARTIAL_TP):
                # Trade Finished
                trades_log.append({
                    'exit_time': cur_time,
                    'type': active_trade.direction,
                    'pnl_pct': float(active_trade.pnl),
                    'status': active_trade.status.name
                })
                active_trade = None
            continue
            
        # 2. Execute Pending Signal
        if pending_signal is not None:
            ps = pending_signal
            pending_signal = None
            dist = label_cfg.sl_multiplier * ps['atr']
            sl = ps['entry'] - dist if ps['dir'] == 'LONG' else ps['entry'] + dist
            
            active_trade = TradeState(
                direction=ps['dir'],
                entry_price=ps['entry'],
                current_stop_loss=sl,
                take_profit=0.0,
                atr_at_entry=ps['atr']
            )
            trades_log.append({
                'entry_time': cur_time,
                'type': ps['dir'],
                'confidence': ps['conf'],
                'atr_pct': ps['atr_pct'],
                'regime_choppy': ps['choppy']
            })
            daily_trades += 1
            continue
            
        # 3. Look for New Signals
        pshort, plong = p_short[i], p_long[i]
        conf = max(pshort, plong)
        
        if conf < confidence_threshold: continue
        if daily_trades >= config.execution.max_trades_per_day: continue
        if config.execution.time_of_day_filter.enabled and (cur_time.hour if hasattr(cur_time, 'hour') else pd.to_datetime(cur_time).hour) in config.execution.time_of_day_filter.blocked_hours_utc: continue
        if adxs[i] < config.execution.min_adx: continue
        if atr_pcts[i] < min_atr_percentile: continue
        
        prob_choppy = regime_probs[i, 2]
        if prob_choppy > config.regime.choppy_threshold: continue
        
        # Signal triggered for next bar OPEN
        direction = 'LONG' if plong > pshort else 'SHORT'
        pending_signal = {
            'dir': direction,
            'entry': opens[i+1],
            'atr': atrs[i],
            'conf': conf,
            'atr_pct': atr_pcts[i],
            'choppy': prob_choppy
        }

    # Clean up logs
    if not trades_log:
        return pd.DataFrame()
        
    # Merge entry/exit dictionaries
    merged = []
    for j in range(0, len(trades_log)-1, 2):
        if 'entry_time' in trades_log[j] and 'exit_time' in trades_log[j+1]:
            d = trades_log[j].copy()
            d.update(trades_log[j+1])
            merged.append(d)
            
    return pd.DataFrame(merged)


## 3. ATR Percentile Grid Search

In [ ]:
results = []
atr_range = np.arange(0.00, 0.35, 0.05)

print("Sweeping ATR Threshold...")
for atr_thresh in tqdm(atr_range):
    trades = simulate_sweep(df, wf_predictions, config, atr_thresh, config.execution.confidence_threshold)
    if len(trades) == 0:
        continue
        
    wins = len(trades[trades['pnl_pct'] > 0])
    gross_pnl = trades['pnl_pct'].sum() * 100
    win_rate = (wins / len(trades)) * 100
    
    results.append({
        'ATR_Percentile': atr_thresh,
        'Trades': len(trades),
        'Win_Rate': win_rate,
        'Gross_PnL_Pct': gross_pnl
    })

res_df = pd.DataFrame(results)
display(res_df)

plt.figure(figsize=(10, 5))
sns.lineplot(data=res_df, x='ATR_Percentile', y='Gross_PnL_Pct', marker='o')
plt.title('Gross PnL vs ATR Percentile Threshold')
plt.grid(True, alpha=0.3)
plt.show()

## 4. Performance by Time of Day (London vs NY)

In [ ]:
# Run one simulation with the BEST ATR from above
optimal_atr = res_df.loc[res_df['Gross_PnL_Pct'].idxmax()]['ATR_Percentile'] if len(res_df) > 0 else 0.15
best_trades = simulate_sweep(df, wf_predictions, config, optimal_atr, config.execution.confidence_threshold)

if len(best_trades) > 0:
    best_trades['hour'] = best_trades['entry_time'].dt.hour
    
    hourly_stats = best_trades.groupby('hour').agg(
        trades=('pnl_pct', 'count'),
        win_rate=('pnl_pct', lambda x: (x > 0).mean() * 100),
        total_pnl=('pnl_pct', lambda x: x.sum() * 100)
    ).reset_index()

    fig, ax1 = plt.subplots(figsize=(12, 6))
    sns.barplot(x='hour', y='total_pnl', data=hourly_stats, alpha=0.6, ax=ax1, color='cyan')
    ax2 = ax1.twinx()
    sns.lineplot(x='hour', y='win_rate', data=hourly_stats, color='yellow', marker='o', ax=ax2)
    ax1.set_title('Performance Heatmap by Hour of Day (UTC)')
    ax1.set_ylabel('Total PnL %')
    ax2.set_ylabel('Win Rate %')
    plt.grid(True, alpha=0.3)
    plt.show()

## 5. Optimal Config Recommendations
Update your `config.yaml` with the findings from this notebook to maximize live trading performance.